# Tune the bootstrap-PF gains with Optuna (full 13-parameter search)

The no-network baseline (`use_net=False`) replaces the 13 controller outputs with
fixed gains. Here we tune **all 13**: one process-noise scale **per state dim** (6),
one prior-regularization weight **per state dim** (6), and a single likelihood
temperature (1). Objective = leave-one-unit-out dev tail-NLL (same metric as
training). Trials are CPU-bound and parallelize across cores via `n_jobs`.

In [1]:
import json
import os

import numpy as np
import optuna
import pandas as pd
import torch

from experiment_config import SEED, DegModel, dataset_paths, pfnet_paths
from src.helpers.seed import set_global_seed
from src.models.particle_filter.core import ParticleFilter
from src.training.pfnet_hparams import PFNET_ARGS


## Configuration

In [ ]:
DATA_NAME = "DS01"
EVAL_REPS = 3
N_TRIALS = 600

In [ ]:
# ARGS_ID here only picks the OUTPUT folder (net_arg0); it does NOT select what
# is optimized. The search always tunes the no-network baseline gains from
# scratch. Paste the printed result into PFNET_ARGS to create the real arg set.
ARGS_ID = 0

# PF + eval settings come from the shared config so tuning matches the pipeline
_args = PFNET_ARGS[ARGS_ID]
N_PARTICLES = int(_args["PARTICLE_FILTER"]["N_PARTICLES"])
MAX_LIFE = int(_args["PARTICLE_FILTER"]["MAX_LIFE"])
LOSS_TAIL_STEPS = int(_args["TRAINING"]["LOSS_TAIL_STEPS"])
SEED_STRIDE = int(_args["EVALUATION"]["SEED_STRIDE"])

STATE_DIM = DegModel.state_dim()  # 6 -> 13 gains (6 noise + 6 prior + 1 lik)


# one torch thread per Optuna worker so threads do not oversubscribe the cores
torch.set_num_threads(1)
N_JOBS = int(os.environ.get("SLURM_CPUS_PER_TASK") or os.cpu_count() or 1)

ESTIMATION_DIR, DEGR_MODEL_DIR = dataset_paths(
    DATA_NAME, fields=["estimation", "degr_model"]
)
_, PRED_DIR = pfnet_paths(ARGS_ID, DATA_NAME)
PRED_DIR.mkdir(parents=True, exist_ok=True)
set_global_seed(SEED)
print(f"n_jobs={N_JOBS}")

n_jobs=16


## Load dev data and degradation models (once)

In [4]:
dev_hi = pd.read_csv(ESTIMATION_DIR / "data_dev.csv")
dev_units = sorted(dev_hi["unit"].astype(int).unique().tolist())
perform_names = [c for c in dev_hi.columns if c not in ["unit", "cycle", "hs"]]


def build_tensors(df, name):
    out = {}
    for u in dev_units:
        sub = df[df["unit"] == u]
        out[u] = torch.tensor(
            np.stack([sub["cycle"].values, sub[name].values], axis=1),
            dtype=torch.float32,
        )
    return out


dev_tensors, dev_degmodels = {}, {}
for name in perform_names:
    dev_tensors[name] = build_tensors(dev_hi, name)
    models = {}
    for u in dev_units:
        m = DegModel()
        m.load_state_dict(
            torch.load(DEGR_MODEL_DIR / "states" / name / f"unit_{u}" / "best_model.pt")
        )
        models[u] = m
    dev_degmodels[name] = models

print("dev units:", dev_units, "| metrics:", perform_names)

dev units: [1, 2, 4, 5, 6] | metrics: ['T48', 'SmFan', 'SmLPC', 'SmHPC']


## Tail-NLL evaluation with tunable gains

In [5]:
def tail_nll(pf, t_data, s_data):
    step_losses = []
    for k in range(len(t_data)):
        mixture = pf.step(t_obs=t_data[[k]], s_obs=s_data[[k]])
        start = -LOSS_TAIL_STEPS if LOSS_TAIL_STEPS else k
        dist = mixture.distribution(s=s_data[start:])
        step_losses.append(-dist.log_prob(t_data[start:]).mean())
    return float(torch.stack(step_losses).mean().item())


@torch.no_grad()
def eval_unit(base_models, unit_tensor, seeds, cn, cp, cl):
    t_data, s_data = unit_tensor[:, 0], unit_tensor[:, 1]
    reps = []
    for sd in seeds:
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(sd)
            pf = ParticleFilter(
                base_models=base_models,
                net=None,
                n_particles=N_PARTICLES,
                max_life=MAX_LIFE,
                use_net=False,
                const_noise=cn,
                const_prior=cp,
                const_lik=cl,
            ).eval()
            reps.append(tail_nll(pf, t_data, s_data))
    return float(np.mean(reps))

## Objective: leave-one-unit-out dev tail-NLL over all metrics

In [6]:
def objective(trial):
    cn = [
        trial.suggest_float(f"noise_{d}", 0.05, 5.0, log=True) for d in range(STATE_DIM)
    ]
    cp = [trial.suggest_float(f"prior_{d}", 0.0, 2.0) for d in range(STATE_DIM)]
    cl = trial.suggest_float("lik", 0.1, 10.0, log=True)
    losses = []
    for name in perform_names:
        for u in dev_units:
            base = [dev_degmodels[name][v] for v in dev_units if v != u]
            seeds = [SEED + u * SEED_STRIDE + r for r in range(EVAL_REPS)]
            losses.append(eval_unit(base, dev_tensors[name][u], seeds, cn, cp, cl))
    return float(np.mean(losses))

## Run the study (parallel, resumable)

In [ ]:
storage = f"sqlite:///{(PRED_DIR / 'optuna_pf_gains_full.db').as_posix()}"
study = optuna.create_study(
    direction="minimize",
    study_name=f"pf_gains_full_{DATA_NAME}",
    storage=storage,
    load_if_exists=True,  # resumable + multi-worker safe
    sampler=optuna.samplers.TPESampler(seed=SEED),
)

# N_TRIALS is the TOTAL target: run only the trials still missing in the db.
# Re-running with the same N_TRIALS just reports; a larger N_TRIALS tops up.
n_done = len(
    study.get_trials(deepcopy=False, states=(optuna.trial.TrialState.COMPLETE,))
)
n_remaining = max(0, N_TRIALS - n_done)
print(
    f"{n_done} completed trials in db; running {n_remaining} more to reach {N_TRIALS}"
)
if n_remaining:
    study.optimize(
        objective, n_trials=n_remaining, n_jobs=N_JOBS, show_progress_bar=True
    )

b = study.best_params
print("best value (mean tail-NLL):", study.best_value)
print("best params:", b)
(PRED_DIR / "optuna_best_gains_full.json").write_text(json.dumps(b, indent=2))

# assemble per-dim vectors and print a ready-to-paste no-network baseline arg
noise = [round(b[f"noise_{d}"], 4) for d in range(STATE_DIM)]
prior = [round(b[f"prior_{d}"], 4) for d in range(STATE_DIM)]
lik = round(b["lik"], 4)
print("\n# add a no-network baseline to PFNET_ARGS (id = multiple of 10):")
print(
    f"    20: make_args(net=None, gains_=gains(noise={noise}, prior={prior}, lik={lik})),"
)


[I 2026-08-05 19:48:57,154] A new study created in RDB with name: pf_gains_full_DS01


  0%|          | 0/600 [00:00<?, ?it/s]

[I 2026-08-05 20:00:13,831] Trial 10 finished with value: 17.494176312287646 and parameters: {'noise_0': 0.18513275015195998, 'noise_1': 0.3941390798231586, 'noise_2': 0.06510690678494967, 'noise_3': 0.9562150642036943, 'noise_4': 0.11389546982724047, 'noise_5': 0.29117592533026976, 'prior_0': 0.0954512423273246, 'prior_1': 1.1250831723768595, 'prior_2': 1.3286684737432046, 'prior_3': 0.7304895051063418, 'prior_4': 0.014322851564660732, 'prior_5': 0.0897871144413751, 'lik': 2.7716589529164413}. Best is trial 10 with value: 17.494176312287646.
[I 2026-08-05 20:00:27,707] Trial 2 finished with value: 15.652081374327343 and parameters: {'noise_0': 0.10427285841137975, 'noise_1': 0.3657202131909591, 'noise_2': 0.39387387661789897, 'noise_3': 0.3506498193652467, 'noise_4': 0.05209036227287047, 'noise_5': 0.05144012073553507, 'prior_0': 0.22476726665132696, 'prior_1': 1.8790111738047774, 'prior_2': 0.6186800407374955, 'prior_3': 1.671465886477271, 'prior_4': 0.4696363109215156, 'prior_5': 1.

## Sanity check: tuned vs default gains

In [ ]:
def dev_score(cn, cp, cl):
    return float(
        np.mean(
            [
                eval_unit(
                    [dev_degmodels[n][v] for v in dev_units if v != u],
                    dev_tensors[n][u],
                    [SEED + u * SEED_STRIDE + r for r in range(EVAL_REPS)],
                    cn,
                    cp,
                    cl,
                )
                for n in perform_names
                for u in dev_units
            ]
        )
    )


b = study.best_params
noise_vec = [b[f"noise_{d}"] for d in range(STATE_DIM)]
prior_vec = [b[f"prior_{d}"] for d in range(STATE_DIM)]
lik = b["lik"]
print("default:", dev_score([1.0] * STATE_DIM, [0.0] * STATE_DIM, 1.0))
print("tuned  :", dev_score(noise_vec, prior_vec, lik))
print("best params:", b)